# Dupin · Fase 6 — Muestra del feed para el dashboard

Genera una muestra del **periodo de test** (steps ≥ 480, superficie TRANSFER/
CASH_OUT) en orden temporal y la publica como JSONL en
`gs://dupin-dupin-artifacts/demo/feed.jsonl`. El serving la sirve vía
`/v1/demo-feed` y el dashboard la reproduce contra la API real.

PaySim es CC BY-SA 4.0 → la muestra vive en GCS (bucket de artifacts, que la SA
de serving ya puede leer), **nunca en git**. Incluye `isFraud` solo como
ground-truth para que el dashboard muestre recall/precisión en vivo.

In [ ]:
# Clonar el repo para usar EL MISMO features/ del entrenamiento (paridad del estado).
from google.colab import userdata
import sys, subprocess
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
subprocess.run(["rm", "-rf", "/content/dupin"])
subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/alexxcode/dupin.git", "/content/dupin"], check=True)
if "/content/dupin" not in sys.path:
    sys.path.insert(0, "/content/dupin")
subprocess.run(["git", "-C", "/content/dupin", "log", "--oneline", "-1"])

## 1. Auth + carga del raw (periodo de test)

In [ ]:
from google.colab import auth
auth.authenticate_user()
import pandas as pd

PROJECT_ID = "dupin-dupin"
BUCKET_RAW = "dupin-dupin-raw"
BUCKET_ART = "dupin-dupin-artifacts"
RAW_URI = f"gs://{BUCKET_RAW}/raw/paysim/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(
    RAW_URI,
    usecols=["step","type","amount","nameOrig","nameDest","isFraud"],
    storage_options={"project": PROJECT_ID},
)
test = df[(df["step"] >= 480) & (df["type"].isin(["TRANSFER","CASH_OUT"]))].copy()
print("Filas en superficie del periodo de test:", len(test))
print("Fraudes:", int(test["isFraud"].sum()))

## 2. Muestrear en orden temporal

Tomamos una muestra manejable para el demo (~4000 tx) conservando el orden por
step y **sobre-representando el fraude** para que el stream sea visualmente rico
(el dashboard usa `isFraud` solo para medir, no para decidir).

In [ ]:
N_LEGIT, N_FRAUD = 3500, 500
frauds = test[test["isFraud"] == 1]
legit  = test[test["isFraud"] == 0]

samp = pd.concat([
    frauds.sample(min(N_FRAUD, len(frauds)), random_state=42),
    legit.sample(min(N_LEGIT, len(legit)), random_state=42),
]).sort_values("step", kind="stable").reset_index(drop=True)

print("Muestra:", len(samp), "| fraudes:", int(samp["isFraud"].sum()))
samp.head(3)

## 3. Publicar como JSONL a GCS

In [ ]:
---
**Listo.** El serving usa dos artefactos del demo:
- `DUPIN_DEMO_FEED_URI=gs://dupin-dupin-artifacts/demo/feed.jsonl` — el feed.
- `DUPIN_WARM_STATE_URI=gs://dupin-dupin-artifacts/demo/warm_state.json.gz` — el
  estado precargado para que el scoring en vivo **no** sea cold-start y coincida
  con la evaluación offline.

Redespliega (o espera a que escale a cero) para que una instancia nueva cargue
ambos. Sin el warm-state, el dashboard arranca cold-start y los receptores salen
como cuentas nuevas, deprimiendo los scores.

In [ ]:
import json
from google.cloud import storage

lines = []
for r in samp.itertuples(index=False):
    lines.append(json.dumps({
        "step": int(r.step), "type": r.type, "amount": float(r.amount),
        "nameOrig": r.nameOrig, "nameDest": r.nameDest, "isFraud": int(r.isFraud),
    }))
payload = "\n".join(lines)

blob = storage.Client(project=PROJECT_ID).bucket(BUCKET_ART).blob("demo/feed.jsonl")
blob.upload_from_string(payload, content_type="application/x-ndjson")
print(f"Publicado: gs://{BUCKET_ART}/demo/feed.jsonl  ({len(lines)} tx)")

---
**Listo.** El serving usa `DUPIN_DEMO_FEED_URI=gs://dupin-dupin-artifacts/demo/feed.jsonl`.
Una instancia nueva de Cloud Run (cold start) cargará este feed; si ya hay una
caliente, redespliega o espera a que escale a cero. Sin este objeto, el dashboard
usa un feed sintético de respaldo.